# TEM Condenser Wide-Area Solver (Optimistix Only)

This notebook solves a TEM condenser setting sweep for wide-area illumination using an **optimistix-only** root solver.

Per target FWHM diameter (50-500 nm, 10 points), it solves for `(f_CL1, f_CL3)` while holding `f_Cmini` fixed and enforcing:

1. `B(S->F_OPL) ~= 0`
2. Target specimen FWHM diameter
3. `alpha < 1 mrad` (acceptance constraint)

The objective pre-field lens (`OPL`) remains fixed at `5 mm`.


In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
_src = _repo_root / 'src'
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp

try:
    import optimistix as optx
except Exception as exc:
    raise ImportError(
        'optimistix is required for this notebook. Install it in the active environment and rerun.'
    ) from exc

from temgym_core.constants import energy2wavelength
from temgym_core.components import Lens, Plane
from temgym_core.gaussian import make_gaussian
from temgym_core.source import make_waist_divergence_rays
from temgym_core.run import run_to_end

print('optimistix available:', True)


## Config And Defaults

All distances are editable and in meters unless labeled otherwise.


In [ ]:
cfg = {
    # Beam and electron optics
    'voltage': 200e3,
    'waist_in': 10e-9,

    # Fixed objective pre-field optics
    'f_OPL': 5.0e-3,
    'd_OPL_Sp': 1.0e-3,

    # Geometry (cm-scale drifts)
    'd_S_C1': 1.2e-2,
    'd_C1_C3': 1.8e-2,
    'd_C3_A': 8.0e-3,
    'd_A_Cmini': 2.5e-2,
    'd_Cmini_OPL': 1.8e-2,

    # Fixed Cmini candidate set in mm (auto-selected by feasibility scoring)
    'f_Cmini_candidates_mm': [6.0, 8.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0],

    # Target specimen FWHM diameters in nm
    'target_fwhm_nm': np.linspace(50.0, 500.0, 10),

    # Bounds and seeds in mm
    'bounds_mm': {
        'f_CL1': (0.8, 15.0),
        'f_CL3': (0.8, 25.0),
    },
    'seed_f_CL1_mm': 2.0,
    'seed_f_CL3_mm': 6.0,

    # Optimistix multistart controls
    'n_multistart_base': 12,
    'n_multistart_retry': 36,
    'multistart_seed': 23,
    'lm_rtol': 1e-11,
    'lm_atol': 1e-11,
    'lm_max_steps': 220,

    # Acceptance tolerances
    'tol_B': 1e-8,
    'tol_fwhm_m': 2e-9,
    'alpha_cap': 1e-3,
}

if cfg['d_Cmini_OPL'] <= cfg['f_OPL']:
    raise ValueError('Need d_Cmini_OPL > f_OPL so the OPL front focal plane is upstream of OPL.')

cfg['f_Cmini_candidates'] = 1e-3 * np.asarray(cfg['f_Cmini_candidates_mm'], dtype=float)
cfg['target_fwhm'] = 1e-9 * np.asarray(cfg['target_fwhm_nm'], dtype=float)

bounds_arr = np.array([
    [1e-3 * cfg['bounds_mm']['f_CL1'][0], 1e-3 * cfg['bounds_mm']['f_CL1'][1]],
    [1e-3 * cfg['bounds_mm']['f_CL3'][0], 1e-3 * cfg['bounds_mm']['f_CL3'][1]],
], dtype=float)
cfg['bounds_arr'] = bounds_arr

cfg['seed_x0'] = np.array([
    1e-3 * cfg['seed_f_CL1_mm'],
    1e-3 * cfg['seed_f_CL3_mm'],
], dtype=float)

cfg['wavelength'] = float(np.asarray(energy2wavelength(cfg['voltage'])))
cfg['theta0'] = cfg['wavelength'] / (np.pi * cfg['waist_in'])

print('wavelength [pm]:', cfg['wavelength'] * 1e12)
print('theta0 [mrad]:', cfg['theta0'] * 1e3)
print('bounds [mm]:', bounds_arr * 1e3)
print('targets [nm]:', cfg['target_fwhm_nm'])


## Optical Chains And Metrics

We use standard paraxial drifts/lenses to build:

- `M(S->F_OPL)` for the `B ~= 0` imaging constraint
- `M(S->Sp)` for Gaussian FWHM and alpha metrics


In [ ]:
def P_np(d):
    return np.array([[1.0, float(d)], [0.0, 1.0]], dtype=float)


def L_np(f):
    return np.array([[1.0, 0.0], [-1.0 / float(f), 1.0]], dtype=float)


def compose_forward_np(ops):
    M = np.eye(2, dtype=float)
    for op in ops:
        M = np.asarray(op, dtype=float) @ M
    return M


def P_jax(d):
    return jnp.array([[1.0, d], [0.0, 1.0]], dtype=jnp.float64)


def L_jax(f):
    return jnp.array([[1.0, 0.0], [-1.0 / f, 1.0]], dtype=jnp.float64)


def compose_forward_jax(ops):
    M = jnp.eye(2, dtype=jnp.float64)
    for op in ops:
        M = op @ M
    return M


def matrix_S_to_A_np(f_cl1, f_cl3, cfg_local):
    return compose_forward_np([
        P_np(cfg_local['d_S_C1']),
        L_np(f_cl1),
        P_np(cfg_local['d_C1_C3']),
        L_np(f_cl3),
        P_np(cfg_local['d_C3_A']),
    ])


def matrix_S_to_FOPL_np(f_cl1, f_cl3, f_cmini, cfg_local):
    return compose_forward_np([
        P_np(cfg_local['d_S_C1']),
        L_np(f_cl1),
        P_np(cfg_local['d_C1_C3']),
        L_np(f_cl3),
        P_np(cfg_local['d_C3_A']),
        P_np(cfg_local['d_A_Cmini']),
        L_np(f_cmini),
        P_np(cfg_local['d_Cmini_OPL'] - cfg_local['f_OPL']),
    ])


def matrix_S_to_Sp_np(f_cl1, f_cl3, f_cmini, cfg_local):
    return compose_forward_np([
        P_np(cfg_local['d_S_C1']),
        L_np(f_cl1),
        P_np(cfg_local['d_C1_C3']),
        L_np(f_cl3),
        P_np(cfg_local['d_C3_A']),
        P_np(cfg_local['d_A_Cmini']),
        L_np(f_cmini),
        P_np(cfg_local['d_Cmini_OPL']),
        L_np(cfg_local['f_OPL']),
        P_np(cfg_local['d_OPL_Sp']),
    ])


def matrix_S_to_FOPL_jax(f_cl1, f_cl3, f_cmini, cfg_local):
    return compose_forward_jax([
        P_jax(cfg_local['d_S_C1']),
        L_jax(f_cl1),
        P_jax(cfg_local['d_C1_C3']),
        L_jax(f_cl3),
        P_jax(cfg_local['d_C3_A']),
        P_jax(cfg_local['d_A_Cmini']),
        L_jax(f_cmini),
        P_jax(cfg_local['d_Cmini_OPL'] - cfg_local['f_OPL']),
    ])


def matrix_S_to_Sp_jax(f_cl1, f_cl3, f_cmini, cfg_local):
    return compose_forward_jax([
        P_jax(cfg_local['d_S_C1']),
        L_jax(f_cl1),
        P_jax(cfg_local['d_C1_C3']),
        L_jax(f_cl3),
        P_jax(cfg_local['d_C3_A']),
        P_jax(cfg_local['d_A_Cmini']),
        L_jax(f_cmini),
        P_jax(cfg_local['d_Cmini_OPL']),
        L_jax(cfg_local['f_OPL']),
        P_jax(cfg_local['d_OPL_Sp']),
    ])


def metrics_from_matrices_np(f_cl1, f_cl3, f_cmini, cfg_local):
    M_fopl = matrix_S_to_FOPL_np(f_cl1, f_cl3, f_cmini, cfg_local)
    B_s_to_fopl = float(M_fopl[0, 1])

    M_sp = matrix_S_to_Sp_np(f_cl1, f_cl3, f_cmini, cfg_local)
    A, B, C, D = float(M_sp[0, 0]), float(M_sp[0, 1]), float(M_sp[1, 0]), float(M_sp[1, 1])

    lam = cfg_local['wavelength']
    qinv_in = 1j * lam / (np.pi * cfg_local['waist_in'] ** 2)
    denom = A + B * qinv_in
    qinv_out = (C + D * qinv_in) / denom

    im_qinv = float(np.imag(qinv_out))
    if im_qinv <= 0.0:
        fwhm = np.nan
    else:
        w = np.sqrt(lam / (np.pi * im_qinv))
        fwhm = np.sqrt(2.0 * np.log(2.0)) * w

    alpha = abs(D) * cfg_local['theta0']
    return {
        'B_SFOPL': B_s_to_fopl,
        'fwhm': float(fwhm),
        'alpha': float(alpha),
        'A_SSp': A,
        'D_SSp': D,
        'qinv_out': complex(qinv_out),
    }


def metrics_for_solver_jax(f_cl1, f_cl3, f_cmini, cfg_local):
    M_fopl = matrix_S_to_FOPL_jax(f_cl1, f_cl3, f_cmini, cfg_local)
    B_s_to_fopl = M_fopl[0, 1]

    M_sp = matrix_S_to_Sp_jax(f_cl1, f_cl3, f_cmini, cfg_local)
    A, B, C, D = M_sp[0, 0], M_sp[0, 1], M_sp[1, 0], M_sp[1, 1]

    lam = jnp.asarray(cfg_local['wavelength'], dtype=jnp.float64)
    qinv_in = 1j * lam / (jnp.pi * cfg_local['waist_in'] ** 2)
    qinv_out = (C + D * qinv_in) / (A + B * qinv_in)

    im_qinv = jnp.imag(qinv_out)
    w = jnp.sqrt(jnp.clip(lam / (jnp.pi * jnp.maximum(im_qinv, 1e-30)), 1e-30, 1e30))
    fwhm_raw = jnp.sqrt(2.0 * jnp.log(2.0)) * w
    fwhm = jnp.where(im_qinv > 0.0, fwhm_raw, 1e6)

    alpha = jnp.abs(D) * (lam / (jnp.pi * cfg_local['waist_in']))
    return B_s_to_fopl, fwhm, alpha


## Optimistix-Only Solve Utilities

No Newton or hand-coded fallback is used. All root solves use `optx.LevenbergMarquardt`.


In [ ]:
def bounded_logistic_to_physical_jax(u, bounds_arr):
    bounds_j = jnp.asarray(bounds_arr, dtype=jnp.float64)
    lo = bounds_j[:, 0]
    hi = bounds_j[:, 1]
    return lo + (hi - lo) * jax.nn.sigmoid(u)


def bounded_logit_from_physical_np(x, bounds_arr):
    b = np.asarray(bounds_arr, dtype=float)
    lo = b[:, 0]
    hi = b[:, 1]
    s = np.clip((np.asarray(x, dtype=float) - lo) / (hi - lo), 1e-9, 1.0 - 1e-9)
    return np.log(s) - np.log1p(-s)


def residual_u_jax(u, args):
    target_fwhm, f_cmini, cfg_local, bounds_arr = args
    x = bounded_logistic_to_physical_jax(u, bounds_arr)
    f_cl1, f_cl3 = x[0], x[1]
    B_s_to_fopl, fwhm, _ = metrics_for_solver_jax(f_cl1, f_cl3, f_cmini, cfg_local)
    return jnp.array([B_s_to_fopl, fwhm - target_fwhm], dtype=jnp.float64)


def run_optx_seed(seed_phys, target_fwhm, f_cmini, cfg_local):
    bounds_arr = cfg_local['bounds_arr']
    lo = bounds_arr[:, 0]
    hi = bounds_arr[:, 1]

    x0 = np.clip(np.asarray(seed_phys, dtype=float), lo, hi)
    u0 = bounded_logit_from_physical_np(x0, bounds_arr)

    args = (float(target_fwhm), float(f_cmini), cfg_local, bounds_arr)
    solver = optx.LevenbergMarquardt(rtol=cfg_local['lm_rtol'], atol=cfg_local['lm_atol'])
    sol = optx.root_find(
        residual_u_jax,
        solver,
        y0=jnp.asarray(u0, dtype=jnp.float64),
        args=args,
        options={'jac': 'fwd'},
        max_steps=int(cfg_local['lm_max_steps']),
        throw=False,
    )

    u_star = np.asarray(sol.value, dtype=float)
    x_star = np.asarray(bounded_logistic_to_physical_jax(u_star, bounds_arr), dtype=float)

    m = metrics_from_matrices_np(x_star[0], x_star[1], f_cmini, cfg_local)
    residual = np.array([m['B_SFOPL'], m['fwhm'] - target_fwhm], dtype=float)
    residual_norm = float(np.linalg.norm(residual))

    success = bool(
        np.isfinite(residual_norm)
        and abs(residual[0]) <= cfg_local['tol_B']
        and abs(residual[1]) <= cfg_local['tol_fwhm_m']
        and m['alpha'] < cfg_local['alpha_cap']
    )

    return {
        'success': success,
        'status': str(sol.result),
        'num_steps': int(np.asarray(sol.stats.get('num_steps', 0))),
        'x': x_star,
        'residual': residual,
        'residual_norm': residual_norm,
        'B_SFOPL': float(m['B_SFOPL']),
        'fwhm': float(m['fwhm']),
        'alpha': float(m['alpha']),
    }


def generate_seed_bank(cfg_local, rng, warm_start=None, n_random=0):
    seeds = []
    if warm_start is not None:
        seeds.append(np.asarray(warm_start, dtype=float))

    base = [
        cfg_local['seed_x0'],
        np.array([2.0e-3, 4.0e-3]),
        np.array([2.0e-3, 8.0e-3]),
        np.array([3.0e-3, 6.0e-3]),
        np.array([5.0e-3, 10.0e-3]),
        np.array([8.0e-3, 6.0e-3]),
    ]
    seeds.extend(base)

    lo = cfg_local['bounds_arr'][:, 0]
    hi = cfg_local['bounds_arr'][:, 1]
    for _ in range(int(n_random)):
        seeds.append(rng.uniform(lo, hi))

    uniq = []
    for s in seeds:
        if not any(np.linalg.norm(np.asarray(s) - np.asarray(u)) < 1e-12 for u in uniq):
            uniq.append(np.asarray(s, dtype=float))
    return uniq


def pick_best_result(results):
    if not results:
        return None
    successes = [r for r in results if r['success']]
    if successes:
        return min(successes, key=lambda r: (r['residual_norm'], r['alpha']))
    return min(results, key=lambda r: (r['residual_norm'], r['alpha']))


def solve_target_optx_only(target_fwhm, f_cmini, cfg_local, warm_start, rng):
    attempts = []

    seeds_base = generate_seed_bank(
        cfg_local,
        rng,
        warm_start=warm_start,
        n_random=cfg_local['n_multistart_base'],
    )
    for seed in seeds_base:
        attempts.append(run_optx_seed(seed, target_fwhm, f_cmini, cfg_local))

    best = pick_best_result(attempts)
    if best is not None and best['success']:
        best['stage'] = 'base'
        best['n_attempts'] = len(attempts)
        return best

    seeds_retry = generate_seed_bank(
        cfg_local,
        rng,
        warm_start=(best['x'] if best is not None else warm_start),
        n_random=cfg_local['n_multistart_retry'],
    )
    for seed in seeds_retry:
        attempts.append(run_optx_seed(seed, target_fwhm, f_cmini, cfg_local))

    best = pick_best_result(attempts)
    best['stage'] = 'retry'
    best['n_attempts'] = len(attempts)
    return best


def solve_sweep_for_fcmini(f_cmini, cfg_local):
    rng = np.random.default_rng(cfg_local['multistart_seed'] + int(round(f_cmini * 1e6)))

    out = []
    warm = cfg_local['seed_x0'].copy()
    for target in cfg_local['target_fwhm']:
        solved = solve_target_optx_only(target, f_cmini, cfg_local, warm_start=warm, rng=rng)
        solved['target_fwhm'] = float(target)
        solved['f_cmini'] = float(f_cmini)
        solved['f_CL1'] = float(solved['x'][0])
        solved['f_CL3'] = float(solved['x'][1])
        if solved['success']:
            warm = solved['x']
        out.append(solved)
    return out


def score_fcmini_results(results):
    ok = [r for r in results if r['success']]
    n_ok = len(ok)
    if n_ok == 0:
        return (10_000, 10_000, 10_000)

    med_cl1 = float(np.median([r['f_CL1'] for r in ok]))
    med_dist_to_2mm = abs(med_cl1 - 2.0e-3)
    max_res = float(np.max([r['residual_norm'] for r in ok]))
    return (-n_ok, med_dist_to_2mm, max_res)


def select_best_fcmini(cfg_local):
    reports = []
    for f_cmini in cfg_local['f_Cmini_candidates']:
        results = solve_sweep_for_fcmini(float(f_cmini), cfg_local)
        score = score_fcmini_results(results)
        reports.append({
            'f_cmini': float(f_cmini),
            'results': results,
            'score': score,
            'n_ok': sum(int(r['success']) for r in results),
        })

    reports_sorted = sorted(reports, key=lambda r: r['score'])
    return reports_sorted, reports_sorted[0]


## Auto-Select Fixed Cmini And Run The 10-Point Sweep

In [ ]:
candidate_reports, best_candidate = select_best_fcmini(cfg)

print('Cmini candidate ranking (best first):')
for c in candidate_reports:
    print(
        f"  f_Cmini={c['f_cmini']*1e3:6.2f} mm | solved={c['n_ok']}/10 | score={c['score']}"
    )

selected_f_cmini = best_candidate['f_cmini']
sweep_results = best_candidate['results']

print('\nSelected fixed f_Cmini [mm]:', selected_f_cmini * 1e3)
print('Solved points:', sum(int(r['success']) for r in sweep_results), '/', len(sweep_results))


## Result Table

In [ ]:
def print_results_table(results):
    header = (
        'target_nm  achieved_nm  f_CL1_mm  f_CL3_mm  f_Cmini_mm  '
        'B_SFOPL      alpha_mrad  res_norm     success  stage    optx_status'
    )
    print(header)
    print('-' * len(header))

    for r in results:
        target_nm = r['target_fwhm'] * 1e9
        achieved_nm = r['fwhm'] * 1e9
        f1_mm = r['f_CL1'] * 1e3
        f3_mm = r['f_CL3'] * 1e3
        fc_mm = r['f_cmini'] * 1e3
        b_val = r['B_SFOPL']
        alpha_mrad = r['alpha'] * 1e3
        resn = r['residual_norm']
        success = str(bool(r['success']))
        stage = r.get('stage', '-')
        status = r.get('status', '-')

        print(
            f"{target_nm:8.1f}  {achieved_nm:11.3f}  {f1_mm:8.3f}  {f3_mm:8.3f}  {fc_mm:10.3f}  "
            f"{b_val: .3e}  {alpha_mrad:10.4f}  {resn: .3e}  {success:>7s}  {stage:>6s}  {status}"
        )


print_results_table(sweep_results)


## TemGym-Style Verification (Representative Solved Points)

For a few solved points (low/mid/high target), compare matrix-derived metrics against direct TemGym propagation.


In [ ]:
def build_tem_components(f_cl1, f_cl3, f_cmini, cfg_local):
    z_s = 0.0
    z_c1 = z_s + cfg_local['d_S_C1']
    z_c3 = z_c1 + cfg_local['d_C1_C3']
    z_cmini = z_c3 + cfg_local['d_C3_A'] + cfg_local['d_A_Cmini']
    z_opl = z_cmini + cfg_local['d_Cmini_OPL']
    z_sp = z_opl + cfg_local['d_OPL_Sp']

    return (
        Lens(z=z_c1, focal_length=float(f_cl1)),
        Lens(z=z_c3, focal_length=float(f_cl3)),
        Lens(z=z_cmini, focal_length=float(f_cmini)),
        Lens(z=z_opl, focal_length=float(cfg_local['f_OPL'])),
        Plane(z=z_sp),
    )


def metrics_from_temgym(f_cl1, f_cl3, f_cmini, cfg_local):
    comps = build_tem_components(f_cl1, f_cl3, f_cmini, cfg_local)

    beam = make_gaussian(
        x=0.0,
        y=0.0,
        dx=0.0,
        dy=0.0,
        z=0.0,
        voltage=cfg_local['voltage'],
        waist_x=cfg_local['waist_in'],
        waist_y=cfg_local['waist_in'],
    )
    out_beam = run_to_end(beam, comps)

    lam = float(np.asarray(out_beam.wavelength))
    qinv_xx = complex(np.asarray(out_beam.Q_inv[0, 0]))
    im_qinv = float(np.imag(qinv_xx))
    if im_qinv <= 0.0:
        fwhm = np.nan
    else:
        w = np.sqrt(lam / (np.pi * im_qinv))
        fwhm = np.sqrt(2.0 * np.log(2.0)) * w

    rays = make_waist_divergence_rays(
        waist=cfg_local['waist_in'],
        voltage=cfg_local['voltage'],
        z=0.0,
        x0=0.0,
        y0=0.0,
    )
    out_rays = run_to_end(rays, comps)
    dx_out = np.asarray(out_rays.dx)
    alpha = float(abs(dx_out[1]))

    return {
        'fwhm': float(fwhm),
        'alpha': alpha,
    }


success_indices = [i for i, r in enumerate(sweep_results) if r['success']]
verification_rows = []
if success_indices:
    picks = sorted(set([
        success_indices[0],
        success_indices[len(success_indices) // 2],
        success_indices[-1],
    ]))

    for idx in picks:
        r = sweep_results[idx]
        m_matrix = metrics_from_matrices_np(r['f_CL1'], r['f_CL3'], r['f_cmini'], cfg)
        m_temgym = metrics_from_temgym(r['f_CL1'], r['f_CL3'], r['f_cmini'], cfg)

        fwhm_abs_err = abs(m_temgym['fwhm'] - m_matrix['fwhm'])
        alpha_abs_err = abs(m_temgym['alpha'] - m_matrix['alpha'])

        verification_rows.append({
            'idx': idx,
            'target_nm': r['target_fwhm'] * 1e9,
            'fwhm_matrix_nm': m_matrix['fwhm'] * 1e9,
            'fwhm_temgym_nm': m_temgym['fwhm'] * 1e9,
            'fwhm_abs_err_nm': fwhm_abs_err * 1e9,
            'alpha_matrix_mrad': m_matrix['alpha'] * 1e3,
            'alpha_temgym_mrad': m_temgym['alpha'] * 1e3,
            'alpha_abs_err_urad': alpha_abs_err * 1e6,
        })

print('verification rows:', len(verification_rows))
for row in verification_rows:
    print(row)


## Validation Checks And Summary

In [ ]:
ok_rows = [r for r in sweep_results if r['success']]
failed_rows = [r for r in sweep_results if not r['success']]

if ok_rows:
    assert all(abs(r['B_SFOPL']) <= cfg['tol_B'] + 1e-15 for r in ok_rows)
    assert all(abs(r['fwhm'] - r['target_fwhm']) <= cfg['tol_fwhm_m'] + 1e-15 for r in ok_rows)
    assert all(r['alpha'] < cfg['alpha_cap'] for r in ok_rows)

if verification_rows:
    assert all(row['fwhm_abs_err_nm'] < 0.5 for row in verification_rows)
    assert all(row['alpha_abs_err_urad'] < 5.0 for row in verification_rows)

if len(ok_rows) == len(sweep_results):
    achieved = np.array([r['fwhm'] for r in ok_rows], dtype=float)
    assert np.all(np.diff(achieved) >= -1e-12)

print('Solved targets      :', len(ok_rows), '/', len(sweep_results))
print('Failed targets      :', len(failed_rows))
if ok_rows:
    print('Max alpha [mrad]    :', max(r['alpha'] for r in ok_rows) * 1e3)
    print('Max |B|             :', max(abs(r['B_SFOPL']) for r in ok_rows))
    print('Max FWHM error [nm] :', max(abs(r['fwhm'] - r['target_fwhm']) for r in ok_rows) * 1e9)
else:
    print('No successful solves for this configuration.')


## Plots

In [ ]:
targets_nm = np.array([r['target_fwhm'] * 1e9 for r in sweep_results], dtype=float)
achieved_nm = np.array([r['fwhm'] * 1e9 for r in sweep_results], dtype=float)
f1_mm = np.array([r['f_CL1'] * 1e3 for r in sweep_results], dtype=float)
f3_mm = np.array([r['f_CL3'] * 1e3 for r in sweep_results], dtype=float)
alpha_mrad = np.array([r['alpha'] * 1e3 for r in sweep_results], dtype=float)
absB = np.array([abs(r['B_SFOPL']) for r in sweep_results], dtype=float)
resn = np.array([max(r['residual_norm'], 1e-20) for r in sweep_results], dtype=float)
ok = np.array([bool(r['success']) for r in sweep_results], dtype=bool)

fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

axs[0, 0].plot(targets_nm, f1_mm, 'o-', label='f_CL1')
axs[0, 0].plot(targets_nm, f3_mm, 's-', label='f_CL3')
axs[0, 0].set_xlabel('Target FWHM [nm]')
axs[0, 0].set_ylabel('Focal length [mm]')
axs[0, 0].set_title('Solved Focal Lengths')
axs[0, 0].grid(True, alpha=0.3)
axs[0, 0].legend()

axs[0, 1].plot(targets_nm, targets_nm, 'k--', label='y=x')
axs[0, 1].plot(targets_nm, achieved_nm, 'o-', label='achieved')
axs[0, 1].scatter(targets_nm[~ok], achieved_nm[~ok], color='r', label='failed')
axs[0, 1].set_xlabel('Target FWHM [nm]')
axs[0, 1].set_ylabel('Achieved FWHM [nm]')
axs[0, 1].set_title('Achieved vs Target FWHM')
axs[0, 1].grid(True, alpha=0.3)
axs[0, 1].legend()

axs[1, 0].plot(targets_nm, alpha_mrad, 'o-', label='alpha')
axs[1, 0].axhline(cfg['alpha_cap'] * 1e3, color='r', ls='--', label='alpha cap (1 mrad)')
axs[1, 0].set_xlabel('Target FWHM [nm]')
axs[1, 0].set_ylabel('alpha [mrad]')
axs[1, 0].set_title('Specimen Semi-Angle Constraint')
axs[1, 0].grid(True, alpha=0.3)
axs[1, 0].legend()

axs[1, 1].semilogy(targets_nm, np.maximum(absB, 1e-20), 'o-', label='|B(S->F_OPL)|')
axs[1, 1].semilogy(targets_nm, np.maximum(resn, 1e-20), 's-', label='residual norm')
axs[1, 1].axhline(cfg['tol_B'], color='k', ls='--', label='|B| tolerance')
axs[1, 1].set_xlabel('Target FWHM [nm]')
axs[1, 1].set_ylabel('Magnitude (log scale)')
axs[1, 1].set_title('Constraint Residuals')
axs[1, 1].grid(True, alpha=0.3)
axs[1, 1].legend()

plt.show()
